In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ellipk, ellipe

In [2]:
#Universal Constants
c=3*10**8
mu_0 = 4*(np.pi)*(10**-7)
epsilon = 8.854*10**-12
h_planck = 6.62607015*10**-34
hbar = (6.62607015*10**-34)/(2*np.pi)
k_b = 1.380649*10**-23
e = 1.60217663*10**-19

In [3]:
def find_effective_dieletric(e_r,d,W):
    '''
    Determines the effective dielectric constant of a
    CPW when there is a single substrate
    Input:
    -   e_r = Dieletric Constant of the substrate[float] 
    -   d   = Height of substrate[m,float]
    -   W   = Width of CPW[m,float]

    Output:
    -   e_e = Effective Dieletric Constant of CPW[float]
    '''
    e_e = ((e_r+1)/(2))+((e_r-1)/(2))*(1/(np.sqrt(1+(12*d/W))))
    return e_e

def find_f0_Conformal_Mapping(w,s,e_r,d,W,l):
    '''
    Determine the resonant frequencies for a 
    CPW resonator using conformal mapping techniques.
    Determines the Inductance (L_l) and Capacitance (C_l)
    per unit length. Uses calculations to determine 
    effective dielectric constant of the CPW. 
    Input:
    -   w   = centre conductor width[m,float]
    -   s   = seperation of centre conductor[m,float]
    -   e_r = dieletric constant of substrate[float]
    -   d   = height of substrate[m,float]
    -   W   = width of CPW[m,float]
    -   l   = length of CPW[m,float]

    Output:
    -   L_l = Magnetic Inductance of CPW per unit length[,float]
    -   C_L = Capcitance of CPW per unit length[F,float]
    -   f0  = Resonant Frequency of CPW[Hz,float]
    
    Depends on:
    -   find_effective_permeability
    '''
    #Find Arguements of the First Kind
    k_0 = (w)/(w+2*s)
    k_prime = np.sqrt(1-k_0**2)

    #Find Effective Permeability
    e_eff = find_effective_dieletric(e_r,d,W)

    #Elleptical Integrals
    L_l = ((mu_0)/(4))*(ellipk(k_prime**2))/(ellipk(k_0**2))
    C_l = 4*epsilon*e_eff*((ellipk(k_0**2))/(ellipk(k_prime**2)))

    #Find Resonant Frequencies
    f0 = ((1/np.sqrt(L_l*C_l))/(4*l))/(10**9)

    return L_l,C_l,f0


### Error on Elleptical Integrals

The error on the arguements for the complete elliptic integral of the first kind are we follows:
$$
\frac{\mathrm{d}k_0}{\mathrm{d}w} = \frac{1}{w+2s} - \frac{w}{\left ( w+2s \right)^2} \hspace{5mm} \frac{\mathrm{d}k_0}{\mathrm{d}s} = \frac{-2w}{\left ( w+2s \right)}
$$
$$
\Delta k_0 = \sqrt{\left(\Delta w  \frac{\mathrm{d}k_0}{\mathrm{d}w} \right)^2 + \left( \Delta s \frac{\mathrm{d}k_0}{\mathrm{d}s} \right)^2}
$$
$$
\Delta k'_0 = \Delta k_0 \left | \frac{k_0}{k'_0} \right |
$$

In [4]:
def delta_k0(w,s,dw,ds):
    '''
    Finds the error on the arguments for the elleptical 
    Integrals 
    Input:
    -   w           = central conductor width[m,float]
    -   s           = seperation of central conductor[m,float]
    -   dw          = error on central conductor width[m,float]
    -   ds          = error on seperation of central conductor[m,float]
    
    Output:
    -   dk_0        = error on k_0 argument[float]
    -   dk_prime    = error on k_prime arguement[float]
    '''
    #Find Arguements of the First Kind
    k_0 = (w)/(w+2*s)
    k_prime = np.sqrt(1-k_0**2)

    #Find Derivatives
    dk_0_dw = (1)/(w+2*s) - (w)/(w+2*s)**2
    dk_0_ds = -(2*w)/(w+2*s)

    #Find errors
    dk_0 = np.sqrt((dw*dk_0_dw)**2+(ds*dk_0_ds)**2)
    dk_prime = k_0/k_prime * dk_0
    return k_0,k_prime,dk_0,dk_prime

Error on the Elleptical Integrals as defined as Scipy are as follows:
$$
K(m) = \int_{0}^{\frac{\pi}{2}} \frac{1}{\sqrt{1-m \mathrm{sin}^2(\theta)}} d\theta \hspace{5mm} m = k^2
$$
$$
\frac{\mathrm{d}K}{\mathrm{d}m} = \frac{E(m) - (1-m)K(m)}{2m(1-m)}
$$
$$
\Delta K = \left | \frac{\mathrm{d}K}{\mathrm{d}m} \right |\Delta m
$$

where $E(m)$ is the complete elleptical integral of the second kind.

In [5]:
def delta_elleptical_int(w,s,dw,ds):
    '''
    Find error on elleptical integrals of the 
    first kind with respect to the arguments 
    previously defined
    Input:
    -   w           = central conductor width[m,float]
    -   s           = seperation of central conductor[m,float]
    -   dw          = error on central conductor width[m,float]
    -   ds          = error on seperation of central conductor[m,float]
    
    Output:
    -   dK_k_0      = error on elleptical integral with respect to k_0 argument[float]
    -   dK_k_prime  = error on elleptical integral with respect to k_prime argument[float]

    Depends on:
    -   delta_k0
    '''
    k_0,k_prime,dk_0,dk_prime=delta_k0(w,s,dw,ds)

    #Convert to m for sci.py
    m_k_0 = k_0**2
    m_k_prime = k_prime**2

    #Find error
    dm_k_0 = m_k_0*2*((dk_0)/(k_0))
    dm_k_prime = m_k_prime*2*((dk_prime)/(k_prime))

    #Propergate Errors
    dK_k_0 = dm_k_0*np.sqrt(((ellipe(m_k_0)-(1-m_k_0)*ellipk(m_k_0))/(2*m_k_0*(1-m_k_0)))**2)
    dK_k_prime = dm_k_prime*np.sqrt(((ellipe(m_k_prime)-(1-m_k_prime)*ellipk(m_k_prime))/(2*m_k_prime*(1-m_k_prime)))**2)

    return dK_k_0, dK_k_prime



The error on the Inductance $L_{l,C}$ and Capacitance $C_{l,C}$ per unit length from conformal mapping is defined as:
$$
\Delta L_{l,C} = L_{l,c} \sqrt{\left( \frac{\Delta K (k'_0)}{K(k'_0)} \right)^2 + \left( \frac{\Delta K (k_0)}{K(k_0)} \right)^2}
$$
$$
\Delta C_{l,C} = C_{l,c} \sqrt{\left( \frac{\Delta K (k'_0)}{K(k'_0)} \right)^2 + \left( \frac{\Delta K (k_0)}{K(k_0)} \right)^2 + \left( \frac{\Delta \epsilon_{eff}}{\epsilon_{eff}} \right)^2} 
$$

In [6]:
def delta_e_eff(e_eff,e_r,d,W,dd,dW):

    dev_d = 3*(1-e_r)/(W)*((1+12*d/W)**-1.5)

    dev_W = 3*d*(e_r-1)/(W**2)*((1+12*d/W)**-1.5)

    de_eff = np.sqrt((dd*dev_d)**2+(dW*dev_W)**2)
    return de_eff



def delta_L_C_conformal_mapping(w,s,e_r,d,W,l,dw,ds,dd,dW):
    '''
    Input:
    -   w       = central conductor width[m,float]
    -   s       = seperation of central conductor[m,float]
    -   e_r     = dieletric constant of substrate[float]
    -   d       = height of substrate[m,float]
    -   W       = width of CPW[m,float]
    -   l       = length of CPW[m,float]
    -   dw      = error on central conductor[m,float]
    -   ds      = error on seperation of central conductor[float]
    -   dd      = error on height of substrate[m,float]
    -   dW      = error on width of CPW[m,float]

    Output:
    -   dL_l    = error on Inductance per unit length [H/m,float]
    -   dC_l    = error on Capcitance per unit length [F/m,float]
    
    Depends on:
    -   find_f0_Conformal_Mapping
    -   delta_elleptical_int
    -   find_effective_dielectric
    -   delta_e_eff
    -   
    '''
    #Apply Conformal Mapping
    L_l,C_l,f0 = find_f0_Conformal_Mapping(w,s,e_r,d,W,l)

    #Find Arguements of the First Kind
    k_0 = (w)/(w+2*s)
    k_prime = np.sqrt(1-k_0**2)

    #Integrals
    K_k_0 = ellipk(k_0**2)
    K_k_prime = ellipk(k_prime**2)

    #Error on integrals
    dK_k_0,dK_k_prime = delta_elleptical_int(w,s,dw,ds)

    #Find Effective Permeability
    e_eff = find_effective_dieletric(e_r,d,W)
    de_eff = delta_e_eff(e_eff,e_r,d,W,dd,dW)

    #Error on Inductance
    dL_l = L_l*np.sqrt((dK_k_0/K_k_0)**2 + (dK_k_prime/K_k_prime)**2)

    #Error on Capcitance
    dC_l = C_l*np.sqrt((dK_k_0/K_k_0)**2 + (dK_k_prime/K_k_prime)**2 + (de_eff/e_eff)**2)

    return dL_l,dC_l

In [7]:
#Resonator Lengths
l1 = 556.1*10**-6
l2 = 636.1*10**-6
l3 = 580.1*10**-6
l4 = 668.1*10**-6
l5 = 607.1*10**-6
l6 = 538.8*10**-6
l_vals = np.array([l1,l2,l3,l4,l5,l6])
dl_vals = np.ones(len(l_vals))*0.2*10**-6

#Substrate Height
h_s = 500*10**-6
dh_s = 1*10**-6

#CPW width
w_CPW = 20*10**-6
dw_CPW = 2*10**-6

#Cryo Sapphire dielectric constant
e_sap = 10.1

#Dimensions of CPW
central_width = 10*(10**-6)
dcentral_width = 1*(10**-6)
seperation = 5*(10**-6)
dseperation = 1*(10**-6)

#Apply Conformal Mapping
L_l_cm,C_l_cm,f0_CM = find_f0_Conformal_Mapping(central_width,seperation,e_sap,h_s,w_CPW,l_vals)

#Find Errors
dL_l, dC_l = delta_L_C_conformal_mapping(central_width,seperation,e_sap,h_s,w_CPW,l_vals,dcentral_width,dseperation,dh_s,dw_CPW)
L_l_vals = L_l_cm*l_vals
dL_l_vals = L_l_vals*np.sqrt((dL_l/L_l_cm)**2+(dl_vals/l_vals)**2) 




### Error on Kinetic Inductance

$$
L_{K,l} = \frac{0.18\hbar R_\square}{k_B T_c}\frac{1}{w}
$$


$$
\Delta L_{K,l} = L_{K,l} \sqrt{\left( \frac{\Delta R_N}{R_N} \right)^2 + \left( \frac{\Delta T_c}{T_c} \right)^2 +\left( \frac{\Delta w}{w} \right)^2}
$$


In [8]:
def find_Kinetic_Inductance_Resistance(w,Tc,R_sq):
    '''
    Determines the Kinetic inductance per unit length
    for a CPW resonator a given thickness and width.
    Input:
    -   w       = central conductor width[m,float]
    -   Tc      = Critical Temperature of Thin Film[K,float]
    -   R_sq    = Normal State Sheet Resistance of conductor [ohms,float]

    Output:
    -   L_K_l   = Kinetic Inductance per unit length [H/m,float]

    '''
    #Calculate Kinetic Inductance per square
    L_K_sq = (hbar*R_sq)/(1.76*np.pi*k_b*Tc)

    #Per unit Length
    L_K_l = L_K_sq/w
    return L_K_l

def delta_Kinetic_Inductance(w,Tc,R_sq,dw,dTc,dR_sq):
    '''
    Find the error on  the Kinetic Inductance per unit 
    Length
    Input:
    -   w       = central conductor width[m,float]
    -   Tc      = Critical Temperature of Thin Film[K,float]
    -   R_sq    = Normal State Sheet Resistance of conductor[ohms,float]
    -   dw      = error on central conductor width[m,float]
    -   dTc     = error on Critical Temperature of Thin Film[K,float]
    -   dR_sq   = error on Normal State Sheet Resistance[ohms,float]

    Output:
    -   L_K_l   = Kinetic Inductance per unit Length[H/m,float]
    -   dL_K_l  = error on Kinetic Inductance per unit Length[H/m,float]
    
    Depends on:
    -   find_Kinetic_Inductance_Resistance
    '''
    L_K_l = find_Kinetic_Inductance_Resistance(w,Tc,R_sq)
    dL_K_l = L_K_l*np.sqrt((dR_sq/R_sq)**2+(dTc/Tc)**2+(dw/w)**2)

    return L_K_l,dL_K_l


In [9]:
# Parameters taken from James Potters Thesis for NbN
thickness = 10*10**-9
dthickness = 1*10**-9
Temp = 10*10**-3
dTemp = 200*10**-3
NbN_Tc = 9
NbN_dTc = 0.1
NbN_lambda_L = 1145*10**-9
NbN_dlambda_L = 10*10**-9
NbN_R_sq= 1.075*10**3
NbN_dR_sq = 0.001*10**3

L_K_l,dL_K_l1 = delta_Kinetic_Inductance(central_width,NbN_Tc,NbN_R_sq,dcentral_width,NbN_dTc,NbN_dR_sq)

### Kinetic Inductane Error based on London Parameter

$$
L_{K,l} = \mu_0\lambda_L^2(T)\frac{1}{A}
$$

$$
\Delta L_{K,l} = L_{K,l} \sqrt{\left( \frac{\Delta \lambda_L}{\lambda_L} \right)^2+ \left( \frac{\Delta t}{t} \right)^2+ \left( \frac{\Delta w}{w} \right)^2}
$$

In [10]:
def find_Kinetic_Inductance_london(t,w,T,Tc,lambda_L_0):
    '''
    Determines the Kinetic inductance of CPW resonator
    of a material identified by varying the charateristic
    qualities of Critical Temperature (Tc) and London
    Peneteration Depth at Zero Temperature (lambda_L_0)

    Input:
    -   l           = Length of CPW resonator[m,float]
    -   t           = Thickness of conductor[m,float]
    -   w           = Width of centre conductor[m,float]
    -   T           = Temperature of System[K,float]
    -   Tc          = Critical Temperature of Material [K,float]
    -   lambda_L_0  = London Peneteration depth of Material at 0K [K,float]

    Output:
    -   L_K         = Kinetic Inductance of CPW resonator[float]
    '''
    #Find London Penetration Depth
    lambda_L = (lambda_L_0)/(np.sqrt(1-(T/Tc)**4))

    #Calculate Kinetic Inductance
    L_K_l = mu_0*(lambda_L**2)*(1/(t*w))
    return L_K_l


def delta_Kinetic_Inductance_london(t,w,T,Tc,lambda_L_0,dt,dw,dT,dTc,dlambda_L_0):
    '''
        Determines the Kinetic inductance of CPW resonator
        of a material identified by varying the charateristic
        qualities of Critical Temperature (Tc) and London
        Peneteration Depth at Zero Temperature (lambda_L_0)

        Input:
        -   l           = Length of CPW resonator[m,float]
        -   t           = Thickness of conductor[m,float]
        -   w           = Width of centre conductor[m,float]
        -   T           = Temperature of System[K,float]
        -   Tc          = Critical Temperature of Material [K,float]
        -   lambda_L_0  = London Peneteration depth of Material at 0K [K,float]
        -   dl          = Error of length of CPW resonator[m,float]
        -   dt          = Error of thickness of conductor[m,float]
        -   dw          = Error of width of centre conductor[m,float]
        -   dT          = Error of temperature of System[K,float]
        -   dTc         = Error of critical Temperature of Material [K,float]
        -   dlambda_L_0 = Error of london Peneteration depth of Material at 0K [K,float]

        Output:
        -   L_K_l       = Kinetic Inductance of CPW resonator per unit length[H/m,float]
        -   dL_K_l      = Error of Kinetic Inductance of CPW resonator per unit length[H/m,float]

        Depends on:
        -   find_Kinetic_Inductance_london 

        
    '''

    L_K_l = find_Kinetic_Inductance_london(t,w,T,Tc,lambda_L_0)

    dL_K_l = L_K_l *np.sqrt((dlambda_L_0/lambda_L_0)**2+(dt/t)**2+(dw/w)**2)

    return L_K_l,dL_K_l

### Find error on final Resonant Frequency Value
$$
L_l = L_{K,l}+L_{C,l}
$$
$$
v_p = \frac{1}{\sqrt{L_l C_l}}
$$
$$
f_0 = \frac{v_p}{4l}
$$
$$
\Delta f_0 = f_0 \sqrt{ \frac{1}{4} \frac{\Delta L_{K,l}^2 +\Delta L_{C,l}^2}{(L_{K,l} +L_{C,l})^2} + \frac{1}{4} \left ( \frac{\Delta C_l}{C_l} \right )^2 + \left ( \frac{\Delta l}{l} \right)^2} 

In [11]:
def delta_frequency(w,s,e_r,d,W,l,Tc,R_sq,dw,ds,dd,dW,dl,dTc,dR_sq):
    '''
    Finds the analyatical error on the resonant frequencies
    of the coplanar waveguides.
    Input:
    -   w       = central conductor width[m,float]
    -   s       = seperation of central conductor[m,float]
    -   e_r     = dieletric constant of substrate[float]
    -   d       = height of substrate[m,float]
    -   W       = width of CPW[m,float]
    -   l       = length of CPW[m,float]
    -   Tc      = Critical Temperature of Thin Film[K,float]
    -   R_sq    = Normal State Sheet Resistance of conductor[ohms,float]
    -   dw      = error on central conductor width[m,float]
    -   ds      = error on seperation of central conductor[float]
    -   dd      = error on height of substrate[m,float]
    -   dW      = error on width of CPW[m,float]
    -   dl      = error on length of CPW[m,float]
    -   dTc     = error on Critical Temperature of Thin Film[K,float]
    -   dR_sq   = error on Normal State Sheet Resistance[ohms,float]

    Output:
    -   f0      = Resonant frequency of CPW [Hz,float]
    -   df0     = error on Resonant frequency of CPW [Hz,float]
    
    Depends on
    -   delta_Kinetic_Inductance
    -   find_f0_Conformal_Mapping
    -   delta_L_C_conformal_mapping
    '''

    L_K_l,dL_K_l = delta_Kinetic_Inductance(w,Tc,R_sq,dw,dTc,dR_sq)

    L_C_l,C_l,f0 = find_f0_Conformal_Mapping(w,s,e_r,d,W,l)
    dL_C_l,dC_l = delta_L_C_conformal_mapping(w,s,e_r,d,W,l,dw,ds,dd,dW)

    f0 = (1/(4*l*np.sqrt((L_C_l+L_K_l)*C_l))) 

    df0 = f0*np.sqrt(0.25*((dL_K_l**2 +dL_C_l**2)/(L_K_l+L_C_l)**2)+0.25*(dC_l/C_l)**2+(dl/l)**2)
    return f0,df0


In [12]:
def delta_frequency_london(w,s,e_r,d,W,l,t,T,Tc,lambda_L_0,dw,ds,dd,dW,dl,dt,dT,dTc,dlambda_L_0):
    '''
    Finds the analyatical error on the resonant frequencies
    of the coplanar waveguides.
    Input:
    -   w           = central conductor width[m,float]
    -   s           = seperation of central conductor[m,float]
    -   e_r         = dieletric constant of substrate[float]
    -   d           = height of substrate[m,float]
    -   W           = width of CPW[m,float]
    -   l           = length of CPW[m,float]
    -   t           = Thickness of conductor[m,float]
    -   T           = Temperature of System[K,float]
    -   Tc          = Critical Temperature of Thin Film[K,float]
    -   lambda_L_0  = London Peneteration depth of Material at 0K [K,float]
    -   dw          = error on central conductor width[m,float]
    -   ds          = error on seperation of central conductor[float]
    -   dd          = error on height of substrate[m,float]
    -   dW          = error on width of CPW[m,float]
    -   dl          = error on length of CPW[m,float]
    -   dt          = error on thickness of conductor[m,float]
    -   dTc         = error on Critical Temperature of Thin Film[K,float]
    -   dlambda_L_0 = error of london Peneteration depth of Material at 0K [K,float]

    Output:
    -   f0          = Resonant frequency of CPW [Hz,float]
    -   df0         = error on Resonant frequency of CPW [Hz,float]
    
    Depends on:
    -   delta_Kinetic_Inductance_london
    -   find_f0_Conformal_Mapping
    -   delta_L_C_conformal_mapping
    '''

    L_K_l,dL_K_l = delta_Kinetic_Inductance_london(t,w,T,Tc,lambda_L_0,dt,dw,dT,dTc,dlambda_L_0)

    L_C_l,C_l,f0 = find_f0_Conformal_Mapping(w,s,e_r,d,W,l)
    dL_C_l,dC_l = delta_L_C_conformal_mapping(w,s,e_r,d,W,l,dw,ds,dd,dW)

    f0 = (1/(4*l*np.sqrt((L_C_l+L_K_l)*C_l))) 

    df0 = f0*np.sqrt(0.25*((dL_K_l**2 +dL_C_l**2)/(L_K_l+L_C_l)**2)+0.25*(dC_l/C_l)**2+(dl/l)**2)
    return f0,df0


In [13]:
f0,error_f0 = delta_frequency(central_width,
                              seperation,
                              e_sap,
                              h_s,
                              w_CPW,
                              l_vals,
                              NbN_Tc,
                              NbN_R_sq,
                              dcentral_width,
                              dseperation,
                              dh_s,
                              dw_CPW,
                              dl_vals,
                              NbN_dTc,
                              NbN_dR_sq)



In [14]:
f0,error_f0 = delta_frequency_london (central_width,
                                        seperation,
                                        e_sap,
                                        h_s,
                                        w_CPW,
                                        l_vals,
                                        thickness,
                                        Temp,
                                        NbN_Tc,
                                        NbN_lambda_L,
                                        dcentral_width,
                                        dseperation,
                                        dh_s,
                                        dw_CPW,
                                        dl_vals,
                                        dthickness,
                                        dTemp,
                                        NbN_dTc,
                                        NbN_dlambda_L)



In [15]:
# Input Data
sim_f0_kin_res = (np.array([5.566,4.680,4.645,4.434,4.162,6.206]))*10**9

#Polyfit
degree = 1

#
coeff_theory = np.polyfit(1/l_vals,f0,degree)
coeff_sim = np.polyfit(1/l_vals,sim_f0_kin_res,degree)
xarr = np.linspace(min(1/l_vals),max(1/l_vals),1001)
fit_theory = np.poly1d(coeff_theory)
fit_sim = np.poly1d(coeff_sim)
yarr_theory = fit_theory(xarr)
yarr_sim = fit_sim(xarr)

plt.figure(figsize=(12,5))
#plt.title('Resonant Frequency Against Length')
plt.xlabel(r'$\frac{1}{l}$ ($m^{-1}$)')
plt.ylabel(r'Resonant Frequency (Hz)')
plt.errorbar(1/l_vals,f0,yerr=error_f0,fmt='.',capsize=4,label='Analytical Results')
plt.plot(1/l_vals,sim_f0_kin_res,'.',label='Simulation Data')
plt.plot(xarr,yarr_theory,'-',label = 'Analytical Fit')
plt.plot(xarr,yarr_sim,'-',label = 'Simulation Fit')
plt.grid()
plt.legend(loc='best')
plt.show()